# Image Annotation and Filtering with YOLOv5

This notebook demonstrates a process for detecting people in images of various sports, filtering those detections based on manual review, and then analyzing the spatial distribution and coverage of the kept detections.

The workflow involves:
1. **Setting up**: Installing necessary libraries, connecting to Google Drive, defining paths, and loading a YOLO model.
2. **Precomputing Detections**: Running the YOLO model on all images to get initial bounding boxes and confidence scores for people.
3. **Manual Review and Filtering**: Using interactive widgets to review the precomputed detections image by image, selecting which bounding boxes to keep, and saving the filtered annotations and an annotated image.
4. **Analyzing Filtered Data**: Loading the saved filtered annotations, computing various statistics about the kept bounding boxes (e.g., number of people, area coverage, locations), and saving these statistics to CSV files for further analysis.

---
## Setup and Model Loading

This section installs the required libraries (`ultralytics`, `opencv-python`, `ipywidgets`, `pandas`, `tqdm`), mounts Google Drive to access files, imports necessary modules, defines input and output directories, and loads a pre-trained YOLO model (`yolo11m.pt`) for object detection. It also creates output directories for each sport.

In [ ]:
# Build list of images per sport
images_by_sport = {s: sorted(sum([glob.glob(f'{BASE_DIR}/{s}/*.{ext}')
                                  for ext in ['jpg','jpeg','png','bmp','tif','tiff']], []))
                   for s in SPORTS}

# Precompute detections (or skip if you want on-demand)
detections = {s: {} for s in SPORTS}  # sport -> {image_path: {'bboxes': [...], 'confs': [...]}}
for s in SPORTS:
    print(f'Precomputing detections for {s}...')
    for p in tqdm(images_by_sport[s]):
        img = cv2.imread(p)
        if img is None:
            continue
        h, w = img.shape[:2]
        res = model.predict(source=img, classes=[0], conf=CONF_THRESH, verbose=False, device=0)
        bboxes, confs = [], []
        if len(res)>0 and res[0].boxes is not None and len(res[0].boxes)>0:
            xyxy = res[0].boxes.xyxy.cpu().numpy()
            cf   = res[0].boxes.conf.cpu().numpy()
            for (x1,y1,x2,y2), c in zip(xyxy, cf):
                # clip to image
                x1 = np.clip(x1, 0, w-1); x2 = np.clip(x2, 0, w-1)
                y1 = np.clip(y1, 0, h-1); y2 = np.clip(y2, 0, h-1)
                if x2>x1 and y2>y1:
                    bboxes.append([float(x1), float(y1), float(x2), float(y2)])
                    confs.append(float(c))
        detections[s][p] = {'bboxes': bboxes, 'confs': confs}
print('Done.')

## Manual Review and Filtering Widgets

This cell sets up interactive widgets using `ipywidgets` to facilitate the manual review and filtering process. It displays images one by one, shows the detected bounding boxes (initially all as 'kept'), and allows the user to select which boxes to keep using a `SelectMultiple` widget. The selected boxes are highlighted in green, and the dropped ones in red. Buttons are provided to save the current selection and move to the next image. The logic for loading images, drawing boxes, parsing selections, and saving the filtered data (JSON and annotated image) is defined in several functions and linked to the widget events.

In [ ]:
# Widgets
sport_dd   = widgets.Dropdown(options=SPORTS, value=SPORTS[0], description='Sport:')
idx_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description='Image #', continuous_update=False)
boxes_selector = widgets.SelectMultiple(options=[], description='Boxes', rows=10)
save_btn   = widgets.Button(description='Save', button_style='success')
next_btn   = widgets.Button(description='Save & Next', button_style='info')
status_out = widgets.Output()
image_out  = widgets.Output()

def current_list():
    s = sport_dd.value
    lst = images_by_sport[s]
    idx_slider.max = max(0, len(lst)-1)
    return lst

def load_image(path):
    img = cv2.imread(path)
    return img

def draw_image_with_numbers(img, bboxes, keep_indices=None):
    vis = img.copy()
    if keep_indices is None:
        keep_indices = range(len(bboxes))
    for i, (x1,y1,x2,y2) in enumerate(bboxes):
        color = (0,255,0) if i in keep_indices else (0,0,255)  # green keep, red dropped
        cv2.rectangle(vis, (int(x1),int(y1)), (int(x2),int(y2)), color, 2)
        cv2.rectangle(vis, (int(x1), max(0,int(y1)-18)), (int(x1)+22, max(0,int(y1)-2)), color, -1)
        cv2.putText(vis, str(i), (int(x1)+3, max(12,int(y1)-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8,6))
    plt.imshow(vis_rgb); plt.axis('off'); plt.title('Green = keep, Red = drop')
    plt.show()

def refresh():
    with image_out:
        clear_output(wait=True)
        s = sport_dd.value
        lst = current_list()
        if not lst:
            print('No images found for', s)
            return
        i = idx_slider.value
        img_path = lst[i]
        data = detections[s].get(img_path, {'bboxes':[], 'confs':[]})
        bboxes, confs = data['bboxes'], data['confs']

        # Fill the selector with all indices, default = all selected
        options = [f'#{k}  conf={confs[k]:.2f}  [{int(b[0])},{int(b[1])},{int(b[2])},{int(b[3])}]'
                   for k,b in enumerate(bboxes)]
        boxes_selector.options = options
        boxes_selector.value = tuple(options)  # select all by default

        img = load_image(img_path)
        # show all as kept initially
        keep_idxs = set(range(len(bboxes)))
        draw_image_with_numbers(img, bboxes, keep_idxs)

def parse_selected_indices():
    # map selected labels back to indices
    idxs = []
    for label in boxes_selector.value:
        # label starts with `#<idx>`
        try:
            sharp = label.split()[0]  # '#3'
            idx = int(sharp[1:])
            idxs.append(idx)
        except Exception:
            pass
    return set(idxs)

def save_current():
    s   = sport_dd.value
    lst = current_list()
    if not lst:
        return
    i = idx_slider.value
    img_path = lst[i]
    img = load_image(img_path)
    data = detections[s].get(img_path, {'bboxes':[], 'confs':[]})
    bboxes, confs = data['bboxes'], data['confs']

    keep_idxs = parse_selected_indices()
    kept_boxes = [bboxes[j] for j in sorted(list(keep_idxs))]
    kept_confs = [confs[j] for j in sorted(list(keep_idxs))]

    # Save JSON
    base = os.path.splitext(os.path.basename(img_path))[0]
    json_path = f'{OUT_DIR}/{s}/filtered_annotations/{base}.json'
    with open(json_path, 'w') as f:
        json.dump({
            'image': img_path,
            'kept_indices': sorted(list(keep_idxs)),
            'boxes_xyxy': kept_boxes,
            'confs': kept_confs
        }, f, indent=2)

    # Save annotated image (kept = green, dropped = red)
    vis = img.copy()
    for k,(x1,y1,x2,y2) in enumerate(bboxes):
        color = (0,255,0) if k in keep_idxs else (0,0,255)
        cv2.rectangle(vis, (int(x1),int(y1)), (int(x2),int(y2)), color, 2)
        cv2.rectangle(vis, (int(x1), max(0,int(y1)-18)), (int(x1)+22, max(0,int(y1)-2)), color, -1)
        cv2.putText(vis, str(k), (int(x1)+3, max(12,int(y1)-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
    out_img_path = f'{OUT_DIR}/{s}/annotated/{base}__filtered.jpg'
    cv2.imwrite(out_img_path, vis)

    with status_out:
        clear_output(wait=True)
        print(f'Saved:\n- {json_path}\n- {out_img_path}\nKept {len(kept_boxes)} of {len(bboxes)} boxes.')

def on_change(_):
    # When you tweak the selection, redraw with green/red
    with image_out:
        clear_output(wait=True)
        s   = sport_dd.value
        lst = current_list()
        if not lst:
            print('No images for', s); return
        i = idx_slider.value
        img_path = lst[i]
        img = load_image(img_path)
        data = detections[s].get(img_path, {'bboxes':[], 'confs':[]})
        bboxes = data['bboxes']
        keep_idxs = parse_selected_indices()
        draw_image_with_numbers(img, bboxes, keep_idxs)

def on_save(btn):
    save_current()

def on_next(btn):
    save_current()
    # advance index
    if idx_slider.value < idx_slider.max:
        idx_slider.value += 1

# Wire callbacks
boxes_selector.observe(on_change, names='value')
save_btn.on_click(on_save)
next_btn.on_click(on_next)

# Layout
ui = widgets.VBox([
    widgets.HBox([sport_dd, idx_slider]),
    image_out,
    boxes_selector,
    widgets.HBox([save_btn, next_btn]),
    status_out
])

display(ui)
refresh()

def _on_sport_change(change):
    if change['name']=='value':
        idx_slider.value = 0
        refresh()
sport_dd.observe(_on_sport_change)
idx_slider.observe(lambda ch: refresh() if ch['name']=='value' else None)

## Analyze Filtered Annotations (Image Summary)

This cell loads the JSON files containing the manually filtered bounding box annotations for each image. For each image, it calculates image-level statistics based on the *kept* bounding boxes, such as the number of people detected, the total area covered by the union of the bounding boxes (union occupancy), the sum and average area of the bounding boxes, and the mean and median confidence scores of the kept boxes. These image-level statistics are compiled into a pandas DataFrame and saved to a CSV file named `filtered_people_image_summary.csv` in the output directory.

In [ ]:
def load_filtered_boxes(json_path):
    with open(json_path, 'r') as f:
        obj = json.load(f)
    return obj['boxes_xyxy']

rows = []
for s in SPORTS:
    jfiles = sorted(glob.glob(f'{OUT_DIR}/{s}/filtered_annotations/*.json'))
    for jp in jfiles:
        info = json.load(open(jp,'r'))
        img_path = info['image']
        boxes = info['boxes_xyxy']
        img = cv2.imread(img_path)
        if img is None: continue
        h,w = img.shape[:2]

        # union occupancy
        mask = np.zeros((h,w), dtype=np.uint8)
        centers, areas = [], []
        for (x1,y1,x2,y2) in boxes:
            x1=int(x1);y1=int(y1);x2=int(x2);y2=int(y2)
            mask[y1:y2, x1:x2] = 1
            centers.append(((x1+x2)/(2*w), (y1+y2)/(2*h)))
            areas.append((x2-x1)*(y2-y1))
        union_frac = mask.mean()

        rows.append({
            'sport': s,
            'image': img_path,
            'n_person_kept': len(boxes),
            'union_frac': float(union_frac),
            'sum_box_area_px': float(np.sum(areas)) if areas else 0.0,
            'avg_box_area_px': float(np.mean(areas)) if areas else 0.0,
            'centers_norm': centers
        })

df_filtered = pd.DataFrame(rows)
csv_filtered = f'{OUT_DIR}/people_coverage_stats_filtered.csv'
df_filtered.to_csv(csv_filtered, index=False)
csv_filtered


## Analyze Filtered Annotations (Box-Level Long Format)

This cell also loads the JSON files with filtered annotations. However, instead of aggregating statistics per image, it creates a 'long format' dataset where each row represents a single *kept* bounding box. For each bounding box, it records the sport, image path, box index, coordinates (x1, y1, x2, y2), width, height, area, normalized center coordinates (cx_norm, cy_norm), and the confidence score. This box-level data is stored in a pandas DataFrame and saved to a CSV file named `filtered_people_boxes_long.csv` in the output directory. This format is useful for analyzing the distribution and characteristics of individual detections across all images and sports.

In [ ]:
import os, glob, json, cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# paths
OUT_DIR = '/content/drive/MyDrive/spotball_eval_out'
SPORTS  = ['basketball', 'soccer', 'volleyball']

img_rows = []
box_rows = []

for s in SPORTS:
    jdir = f'{OUT_DIR}/{s}/filtered_annotations'
    jfiles = sorted(glob.glob(f'{jdir}/*.json'))
    print(f'{s}: {len(jfiles)} JSON files')

    for jp in tqdm(jfiles, desc=s):
        with open(jp, 'r') as f:
            ann = json.load(f)

        img_path = ann.get('image')
        boxes    = ann.get('boxes_xyxy', [])
        confs    = ann.get('confs', [None]*len(boxes))

        # read image to derive size + union coverage
        img = cv2.imread(img_path) if img_path else None
        if img is not None:
            h, w = img.shape[:2]
        else:
            h, w = None, None

        # compute union occupancy & area stats (only if we have the image)
        union_area_px = None
        union_frac = None
        sum_area = 0.0
        avg_area = 0.0

        if img is not None and len(boxes) > 0:
            mask = np.zeros((h, w), dtype=np.uint8)
            areas = []
            for (x1, y1, x2, y2) in boxes:
                x1 = int(round(x1)); y1 = int(round(y1))
                x2 = int(round(x2)); y2 = int(round(y2))
                x1 = max(0, min(x1, w-1)); x2 = max(0, min(x2, w-1))
                y1 = max(0, min(y1, h-1)); y2 = max(0, min(y2, h-1))
                if x2 > x1 and y2 > y1:
                    mask[y1:y2, x1:x2] = 1
                    areas.append((x2 - x1) * (y2 - y1))
            union_area_px = int(mask.sum())
            union_frac = union_area_px / float(h * w) if (h and w) else None
            sum_area = float(np.sum(areas)) if areas else 0.0
            avg_area = float(np.mean(areas)) if areas else 0.0

        # image-level row
        img_rows.append({
            'sport': s,
            'image': img_path,
            'n_person_kept': len(boxes),
            'img_w': w, 'img_h': h,
            'union_area_px': union_area_px,
            'union_frac': union_frac,          # fraction of image covered by players (union)
            'sum_box_area_px': sum_area,       # naive sum (overlaps counted)
            'avg_box_area_px': avg_area,
            'mean_conf': float(np.mean(confs)) if len(confs) else None,
            'median_conf': float(np.median(confs)) if len(confs) else None,
        })

        # box-level rows (long format)
        for idx, ((x1, y1, x2, y2), conf) in enumerate(zip(boxes, confs)):
            # normalized centers (if image available)
            if img is not None:
                cx_norm = ((x1 + x2) / 2.0) / w
                cy_norm = ((y1 + y2) / 2.0) / h
            else:
                cx_norm = None; cy_norm = None
            box_rows.append({
                'sport': s,
                'image': img_path,
                'box_idx': idx,
                'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                'w': x2 - x1, 'h': y2 - y1,
                'area_px': (x2 - x1) * (y2 - y1),
                'cx_norm': cx_norm, 'cy_norm': cy_norm,
                'conf': conf,
            })

# build dataframes and write
df_img  = pd.DataFrame(img_rows)
df_box  = pd.DataFrame(box_rows)

csv_img = f'{OUT_DIR}/filtered_people_image_summary.csv'
csv_box = f'{OUT_DIR}/filtered_people_boxes_long.csv'
df_img.to_csv(csv_img, index=False)
df_box.to_csv(csv_box, index=False)

print('Wrote:')
print('-', csv_img, f'({len(df_img)} rows)')
print('-', csv_box, f'({len(df_box)} rows)')


## Precompute Detections

This cell iterates through all images in the input directories for each sport, uses the loaded YOLO model to predict bounding boxes for objects belonging to class `0` (which corresponds to 'person' in the COCO dataset the model was trained on) with a confidence threshold of `0.30`. The detected bounding boxes and their confidence scores are stored in a dictionary `detections`, organized by sport and image path. This step can be time-consuming depending on the number and size of images and the chosen YOLO model.

In [ ]:
!pip -q install ultralytics opencv-python ipywidgets pandas tqdm
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, cv2, numpy as np, pandas as pd
from tqdm import tqdm
from ultralytics import YOLO
from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt

# Paths
BASE_DIR = '/content/drive/MyDrive/spotball_eval'      # input root with subfolders: basketball, soccer, volleyball
OUT_DIR  = '/content/drive/MyDrive/spotball_eval_out'  # all outputs here
SPORTS   = ['basketball', 'soccer', 'volleyball']

for s in SPORTS:
    os.makedirs(f'{OUT_DIR}/{s}/annotated', exist_ok=True)
    os.makedirs(f'{OUT_DIR}/{s}/filtered_annotations', exist_ok=True)  # JSON with kept boxes

# Model
model = YOLO('yolo11m.pt')  # or 'yolo11x.pt' if you have GPU headroom
CONF_THRESH = 0.30